# Baseline Models

Before building the "real" recommender (ALS), we need honest baselines to measure it against — otherwise a metric like "12% precision" means nothing in isolation.

Two baselines, in increasing order of sophistication:
1. **Popularity** — same top-10 for every user, zero personalization. Sets the floor.
2. **Item-Item Collaborative Filtering** — recommends anime with similar "audience overlap" to what a user already liked. Real personalization, but a simpler/older technique than ALS.

Both are evaluated with **Precision@10** and **Recall@10** on the same test set, so their numbers are directly comparable to each other and to ALS later.

In [3]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix, save_npz, load_npz
from sklearn.metrics.pairwise import cosine_similarity
import os
import kagglehub

In [4]:
DATA_DIR = os.path.join("..", "data")  # same relative-path pattern as before

train = pd.read_parquet(os.path.join(DATA_DIR, "train_ratings.parquet"))
test = pd.read_parquet(os.path.join(DATA_DIR, "test_ratings.parquet"))

#Runtime for Baseline 2 may take awhile with full dataset if so utilize a smaller sample of the dataset to test the baselines

#train = train.sample(5_000_000, random_state=42) 
#test = test.sample(1_000_000, random_state=42)

print(train.shape, test.shape)

(118387327, 4) (29596832, 4)


## Baseline 1 (K most popular shows)

In [10]:
# Count positive interactions per anime, using train only
popularity = train[train['is_positive'] == 1].groupby('anime_id').size().sort_values(ascending=False)

#K most popular shows
K = 10
top_k_popular = popularity.head(K).index.tolist()

print(top_k_popular)

#Find the precision and recall rate
def precision_recall_at_k(test_df, recommended_items, k):
    total_relevant = 0
    total_recommended_relevant = 0
    
    for user_id, group in test_df[test_df['is_positive'] == 1].groupby('user_id'):
        actual_positive = set(group['anime_id'])
        recommended = set(recommended_items[:k])
        
        hits_this_user = len(actual_positive & recommended)
        total_recommended_relevant += hits_this_user
        total_relevant += len(actual_positive)
    
    # Recall: of everything the user actually liked, what fraction did we recommend?
    recall = total_recommended_relevant / total_relevant if total_relevant > 0 else 0
    
    # Precision: of everything we recommended, what fraction did users actually like?
    num_users = test_df['user_id'].nunique()
    precision = total_recommended_relevant / (num_users * k)
    
    return precision, recall

precision, recall = precision_recall_at_k(test, top_k_popular, K)
print(f"Recall@{K}: {recall:.4f}")
print(f"Precision@{K}: {precision:.4f}")


[20, 2, 92, 2376, 99, 726, 100, 1147, 1160, 1167]
Recall@10: 0.0673
Precision@10: 0.0678


## Baseline 2 (Item-Item Collabritive Filtering)

### Building sparse matrix indices

`anime_id`/`user_id` are real MAL IDs — large and non-sequential (not gapless from 0), which sparse matrices require. This cell builds a translation layer: sequential `anime_idx`/`user_idx` (0, 1, 2...) for matrix positions, plus reverse-lookup dictionaries (`anime_id_map`, etc.) to translate back to real IDs later.

**Important:** `test`'s `anime_idx`/`user_idx` are built via `.map(..._reverse)` against `train`'s categories — NOT by copying `train`'s `.cat.codes` directly (that would misalign by row index, since train/test have different, non-overlapping row indices, and silently produce all-NaN columns). Any user/anime in `test` that never appears in `train` will correctly map to `NaN` here — `test_clean` (below) drops those before evaluation, since the model has no way to have learned about them.

In [5]:
anime_ids = train['anime_id'].astype('category')
anime_id_map = dict(enumerate(anime_ids.cat.categories))      
anime_id_map_reverse = {v: k for k, v in anime_id_map.items()} 

user_ids = train['user_id'].astype('category')
user_id_map = dict(enumerate(user_ids.cat.categories))
user_id_map_reverse = {v: k for k, v in user_id_map.items()}

train['anime_idx'] = anime_ids.cat.codes
train['user_idx'] = user_ids.cat.codes

test['anime_idx'] = test['anime_id'].map(anime_id_map_reverse)
test['user_idx'] = test['user_id'].map(user_id_map_reverse)


p_train = train[train['is_positive']==1]
test_clean = test.dropna(subset=['anime_idx', 'user_idx'])

train.to_parquet(os.path.join(DATA_DIR, "train_ratings.parquet"))
test.to_parquet(os.path.join(DATA_DIR, "test_ratings.parquet"))

### Sparse matrix + item-item similarity

Builds a sparse `(anime × user)` matrix from positive interactions only, then computes cosine similarity between every pair of anime — i.e., "how much do their audiences overlap." `dense_output=False` keeps the result sparse where possible, since most anime pairs share zero users.

**Memory note:** at full scale (~20K anime), the resulting similarity matrix has ~144M stored values. Both matrices are saved to disk (`.npz`) immediately after computing, since this step took ~4 minutes and is expensive to redo if the kernel crashes or is restarted.

In [7]:
item_user_matrix = csr_matrix(
    (
    [1] * len(p_train), (p_train['anime_idx'], p_train['user_idx'])
    ),
    shape = (len(anime_id_map), len(user_id_map))
)

item_similarity = cosine_similarity(item_user_matrix, dense_output=False)
save_npz(os.path.join(DATA_DIR, "item_user_matrix.npz"), item_user_matrix)
save_npz(os.path.join(DATA_DIR, "item_similarity.npz"), item_similarity)

### Generating recommendations for one user

For a given user, sums similarity scores across everything they've positively rated, then ranks all anime by that combined score. Already-rated anime are forced to `-1` so they can never be re-recommended. This is a single-user sanity check before running full evaluation below — confirms the pipeline returns real, sensible anime rather than garbage.

In [8]:
def recommend_for_user(user_idx, k=10):
    # Get this user's positively-rated anime (as matrix indices)
    user_rated = p_train[p_train['user_idx'] == user_idx]['anime_idx'].tolist()
    
    if not user_rated:
        return []  
    

    scores = np.asarray(item_similarity[user_rated].sum(axis=0)).flatten()
    
    scores[user_rated] = -1
    
    top_idx = scores.argsort()[::-1][:k]
    
    return [anime_id_map[i] for i in top_idx]

sample_user_idx = 5
recs = recommend_for_user(sample_user_idx, k=10)

path = kagglehub.dataset_download("ramazanturann/user-animelist-dataset")
animes = pd.read_csv(os.path.join(path, "animes.csv"))

print(f'10 Recommend animes: \n\n {animes[animes['animeID'].isin(recs)][['animeID', 'title']]}')

10 Recommend animes: 

      animeID                                    title
11        12                             Cowboy Bebop
37        38                         Samurai Champloo
112      113                                Fate/Zero
114      115                           Bakemonogatari
147      148  Code Geass: Lelouch of the Rebellion R2
222      223                  Neon Genesis Evangelion
282      283                    Great Teacher Onizuka
399      400                            Gurren Lagann
410      411                                 Baccano!
717      718                       Fate/Zero Season 2


### Full evaluation

Computes Precision@10/Recall@10 across a sample of real users (`sample_users=2000` — evaluating all ~350K+ test users individually would be slow, and 2000 gives a statistically stable estimate). Uses `test_clean`, not raw `test` — critical, since `test` still contains the unmapped `NaN` rows described above, which would otherwise make `groupby('user_idx')` fail or silently skip data.

**Result (full 118M-row dataset): Precision@10 = 0.1735, Recall@10 = 0.1645** — substantially ahead of the popularity baseline above (0.0678 / 0.0673), confirming real personalized similarity has genuine value once given enough data.

In [9]:
def precision_recall_itemcf(test_df, k=10, sample_users=None):
    total_relevant = 0
    total_recommended_relevant = 0
    
    test_positive = test_df[test_df['is_positive'] == 1]
    grouped = test_positive.groupby('user_idx')
    
    sample_uid = list(grouped.groups.keys())[0]

    print("Recommended:", recommend_for_user(sample_uid, k=10))
    print("Actually liked (test):", grouped.get_group(sample_uid)['anime_id'].tolist())
    
    users_to_eval = list(grouped.groups.keys())
    if sample_users:
        users_to_eval = np.random.choice(users_to_eval, size=sample_users, replace=False)
    
    for user_idx in users_to_eval:
        group = grouped.get_group(user_idx)
        actual_positive = set(group['anime_id'])
        
        recommended = set(recommend_for_user(user_idx, k=k))  # ← computed fresh, per user
        
        hits_this_user = len(actual_positive & recommended)
        total_recommended_relevant += hits_this_user
        total_relevant += len(actual_positive)
    
    recall = total_recommended_relevant / total_relevant if total_relevant > 0 else 0
    precision = total_recommended_relevant / (len(users_to_eval) * k)
    
    return precision, recall

precision_recall_itemcf(test_clean, k=10, sample_users=2000)

Recommended: [144, 400, 20, 148, 1138, 1147, 86, 971, 223, 115]
Actually liked (test): [16, 23, 100, 41, 101]


(0.1735, 0.16454075584427902)